# Exploratory Notebook Running the `pylitterbot` package

In [4]:
import sys
from pathlib import Path

# Add parent directory to path so we can import config
sys.path.insert(0, str(Path.cwd().parent))

from pylitterbot import Account, LitterRobot3

from config import Config
config = Config()
username = config.credentials.username
password = config.credentials.password

In [5]:
from pylitterbot import Account, LitterRobot3

account = Account(robot_types=[LitterRobot3])

await account.connect(
    username=username,
    password=password,
    load_robots=True,
)

In [6]:
import inspect

async def get_full_usage_history(robot):
    """
    Return the full stored usage/activity history for a Litter Robot.

    Some pylitterbot versions expose this as:
      - robot.get_usage_history()
      - robot.get_activity_history()
      - robot.get_activities()
    """
    candidate_names = (
        "get_usage_history",
        "get_activity_history",
        "get_activities",
    )

    for name in candidate_names:
        method = getattr(robot, name, None)
        if method is None:
            continue

        try:
            result = method()
            if inspect.isawaitable(result):
                result = await result

            if result is None:
                continue

            if isinstance(result, dict):
                for key in ("history", "usage_history", "activities", "items", "data"):
                    if key in result:
                        result = result[key]
                        break

            if isinstance(result, (list, tuple)):
                return list(result)

            if hasattr(result, "__iter__"):
                return list(result)

        except Exception:
            # try the next API name
            continue

    # Fallback: check common properties
    for name in ("history", "usage_history", "activity_history", "activities"):
        value = getattr(robot, name, None)
        if value is not None:
            if isinstance(value, (list, tuple)):
                return list(value)
            return [value]

    raise RuntimeError("No stored usage history API was found on this robot object.")

In [7]:
robot = account.robots[0]

history = await get_full_usage_history(robot)

for item in history:
    print(item)

2026-08-16T01:17:25.053446+00:00: Ready
2026-08-16T00:17:21.752342+00:00: Clean Cycle Complete
2026-08-16T00:15:07.648401+00:00: Clean Cycle In Progress
2026-08-16T00:02:14.132442+00:00: Cat Sensor Timing
2026-08-15T23:18:46.787065+00:00: Ready
2026-08-15T10:18:41.743339+00:00: Clean Cycle Complete
2026-08-15T10:16:25.658142+00:00: Clean Cycle In Progress
2026-08-15T10:08:58.414150+00:00: Cat Sensor Timing
2026-08-15T09:23:50.999484+00:00: Ready
2026-08-14T23:23:48.187970+00:00: Clean Cycle Complete
2026-08-14T23:21:33.987666+00:00: Clean Cycle In Progress
2026-08-14T23:14:08.691323+00:00: Cat Sensor Timing
2026-08-14T23:00:02.971670+00:00: Ready
2026-08-14T17:59:58.363878+00:00: Clean Cycle Complete
2026-08-14T17:57:43.262403+00:00: Clean Cycle In Progress
2026-08-14T17:49:56.800427+00:00: Cat Sensor Timing
2026-08-14T17:31:51.064280+00:00: Ready
2026-08-14T12:31:47.209468+00:00: Clean Cycle Complete
2026-08-14T12:29:31.049795+00:00: Clean Cycle In Progress
2026-08-14T12:22:05.631565+

In [ ]:
# Query the stored usage history in SQLite
import sqlite3
from pathlib import Path

# Add parent directory to the import path so we can import our app code
import sys
sys.path.insert(0, str(Path.cwd().parent))

from usage_store import get_db_path

conn = sqlite3.connect(get_db_path())
conn.row_factory = sqlite3.Row

# Show the most recent 20 stored events
rows = conn.execute(
    """
    SELECT robot_id, event_type, event_time, raw_json
    FROM usage_history
    ORDER BY event_time DESC
    LIMIT 20
    """
).fetchall()

for row in rows:
    print(dict(row))

# Show counts by day for the last 14 days
print("\nCounts by day:")
for row in conn.execute(
    """
    SELECT date(event_time) AS day, COUNT(*) AS total
    FROM usage_history
    WHERE event_time >= datetime('now', '-14 days')
    GROUP BY date(event_time)
    ORDER BY day DESC
    """
):
    print(dict(row))

# Check for clean cycle alerts manually
print("\nClean cycle count for today in CST:")
for row in conn.execute(
    """
    SELECT COUNT(*) AS total
    FROM usage_history
    WHERE event_type LIKE '%clean%cycle%complete%'
      AND date(event_time) = date('now')
    """
):
    print(dict(row))

conn.close()

language":"python", 